# 02 — SwiGLU feed-forward (M1.2)

The MLP inside every Qwen3 decoder block. This notebook imports the tested module
`qwen3_scratch.layers.SwiGLUFeedForward`; nothing is re-implemented here.
CPU only, a few seconds, no downloads. Plots need `matplotlib`
(`pip install -e ".[notebooks]"`); without it a text table is printed instead.

Full write-up: `docs/components/swiglu.md`.

## 1. Motivation

Attention mixes information **between** tokens. The feed-forward network (FFN)
then transforms each token **independently**, and holds most of a Transformer's
parameters (in Qwen3-0.6B: 9.4M per layer for the MLP vs 6.3M for attention).

| variant | formula | projections |
|---|---|---|
| ReLU FFN (original Transformer) | $\mathrm{ReLU}(xW_1)W_2$ | 2 |
| GELU FFN (GPT-2, BERT) | $\mathrm{GELU}(xW_1)W_2$ | 2 |
| GLU (Dauphin et al. 2017) | $(\sigma(xW_g)\odot xW_u)W_d$ | 3 |
| **SwiGLU** (Shazeer 2020; Llama, Qwen) | $(\mathrm{SiLU}(xW_g)\odot xW_u)W_d$ | 3 |

A **gated** unit computes two things from the same input: a *value* (`up`) and a
*gate* (`gate`) deciding, per hidden unit and per token, how much of the value
passes. The product makes the layer multiplicative (roughly quadratic in $x$),
which empirically improves quality per parameter (Shazeer, *GLU Variants Improve
Transformer*, 2020).

## 2. Mathematics and shapes

$$\mathrm{SiLU}(z)=z\,\sigma(z)=\frac{z}{1+e^{-z}}$$

$$\text{gate}=xW_{\text{gate}}^\top,\quad \text{up}=xW_{\text{up}}^\top,\quad
\text{hidden}=\mathrm{SiLU}(\text{gate})\odot\text{up},\quad
\text{out}=\text{hidden}\,W_{\text{down}}^\top$$

| tensor | shape | Qwen3-0.6B |
|---|---|---|
| input `x` | `[B, T, H]` | H = 1024 |
| `gate_proj.weight`, `up_proj.weight` | `[I, H]` | 3072 × 1024 |
| `gate`, `up`, `hidden` | `[B, T, I]` | I = 3072 |
| `down_proj.weight` | `[H, I]` | 1024 × 3072 |
| output | `[B, T, H]` | |

Parameters: $3HI$ (no biases). FLOPs per token: about $2\cdot 3HI$ for the three
matmuls plus $O(I)$ for SiLU and the product.

## 3. SiLU vs ReLU vs GELU

In [ ]:
import torch
import torch.nn.functional as F

z = torch.linspace(-4, 3, 401)
curves = {"ReLU": F.relu(z), "GELU": F.gelu(z), "SiLU": F.silu(z)}

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is None:
    print(f"{'z':>6}" + "".join(f"{k:>10}" for k in curves))
    fns = {"ReLU": F.relu, "GELU": F.gelu, "SiLU": F.silu}
    for v in (-4.0, -2.0, -1.2785, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0):
        zt = torch.tensor(v)
        print(f"{v:6.2f}" + "".join(f"{f(zt).item():10.4f}" for f in fns.values()))
else:
    colors = {"ReLU": "#2a78d6", "GELU": "#eb6834", "SiLU": "#1baf7a"}
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for name, y in curves.items():
        ax.plot(z, y, lw=2, color=colors[name], label=name)
    ax.axhline(0, color="#9a9a94", lw=0.8)
    ax.grid(alpha=0.25, lw=0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_xlabel("z")
    ax.set_ylabel("activation(z)")
    ax.set_title("SiLU is smooth and dips below zero (min ≈ -0.278 at z ≈ -1.28)")
    ax.legend(frameon=False, loc="upper left")
    plt.show()

Why SiLU: it is smooth (non-zero gradient for negative inputs, unlike ReLU),
self-gated ($z\cdot\sigma(z)$), non-monotonic near zero, and cheap. In SwiGLU it
acts as a *soft gate*: large positive `gate` → passes `up` almost linearly,
large negative → closes it.

## 4. Manual calculation (H = 2, I = 3)

In [ ]:
import math

from qwen3_scratch.layers import SwiGLUFeedForward

x = torch.tensor([1.0, -1.0], dtype=torch.float64)
mlp = SwiGLUFeedForward(2, 3, dtype=torch.float64)
with torch.no_grad():
    mlp.gate_proj.weight.copy_(torch.tensor([[1.0, 0.0], [1.0, 1.0], [0.0, -2.0]]))
    mlp.up_proj.weight.copy_(torch.tensor([[2.0, 0.0], [0.0, 1.0], [1.0, 0.0]]))
    mlp.down_proj.weight.copy_(torch.tensor([[1.0, 1.0, 1.0], [0.5, -1.0, 0.0]]))

gate = mlp.gate_proj(x)
up = mlp.up_proj(x)
hidden = F.silu(gate) * up
print("gate   =", gate.tolist())
print("up     =", up.tolist())
print("SiLU   =", F.silu(gate).tolist(), " (1*sigma(1), 0, 2*sigma(2))")
print("hidden =", hidden.tolist())
print("out    =", mlp(x).tolist())

sig1 = 1 / (1 + math.exp(-1))
silu2 = 2 / (1 + math.exp(-2))
print("by hand:", [2 * sig1 + silu2, 0.5 * 2 * sig1])

Unit 2 has `gate = 0`, so `SiLU(0) = 0` closes it completely even though `up = -1`.
Unit 3 has a strong gate (`SiLU(2) ≈ 1.76`) and passes its value.

## 5. Implementation walkthrough

In [ ]:
import inspect

from qwen3_scratch.config import QWEN3_0_6B_BASE

print(inspect.getsource(SwiGLUFeedForward.forward))

big = SwiGLUFeedForward.from_config(QWEN3_0_6B_BASE, device="meta")  # shapes only
for name, p in big.named_parameters():
    print(f"{name:18s} {tuple(p.shape)}")
print("parameters per layer:", sum(p.numel() for p in big.parameters()))

## 6. Verification against references

In [ ]:
from qwen3_scratch.layers import swiglu_textbook

torch.manual_seed(0)
small = SwiGLUFeedForward(64, 192)
xb = torch.randn(2, 7, 64)
ref64 = swiglu_textbook(
    xb.double(),
    small.gate_proj.weight.double(),
    small.up_proj.weight.double(),
    small.down_proj.weight.double(),
)
print("float32 vs float64 textbook, max abs err:", (small(xb).double() - ref64).abs().max().item())

try:
    import transformers
    from transformers.models.qwen3.modeling_qwen3 import Qwen3MLP
except ImportError:
    print("transformers not installed -> HF comparison skipped (pip install -e '.[hf]')")
else:
    hf = Qwen3MLP(transformers.Qwen3Config(hidden_size=64, intermediate_size=192))
    hf.load_state_dict(small.state_dict())  # identical parameter names
    with torch.no_grad():
        print("bit-identical to HF Qwen3MLP:", torch.equal(small(xb), hf(xb)))

## 7. Gradients through both branches

With upstream gradient $u=\partial L/\partial\text{out}$ and
$\delta=u\,W_{\text{down}}$ (shape `[.., I]`):

$$\frac{\partial L}{\partial\text{up}}=\delta\odot\mathrm{SiLU}(\text{gate}),\qquad
\frac{\partial L}{\partial\text{gate}}=\delta\odot\text{up}\odot\mathrm{SiLU}'(\text{gate}),\quad
\mathrm{SiLU}'(z)=\sigma(z)\,(1+z(1-\sigma(z)))$$

Each branch's gradient is scaled by the *other* branch's activation: the gate
learns where the value is useful, the value learns what to send when the gate is open.

In [ ]:
torch.manual_seed(1)
m = SwiGLUFeedForward(8, 20, dtype=torch.float64)
xg = torch.randn(4, 8, dtype=torch.float64, requires_grad=True)
u = torch.randn(4, 8, dtype=torch.float64)
m(xg).backward(u)

with torch.no_grad():
    g, v = m.gate_proj(xg), m.up_proj(xg)
    s = torch.sigmoid(g)
    delta = u @ m.down_proj.weight
    d_gate = delta * v * s * (1 + g * (1 - s))
    d_up = delta * F.silu(g)
    dx = d_gate @ m.gate_proj.weight + d_up @ m.up_proj.weight
print("max |autograd - formula| dx       :", (xg.grad - dx).abs().max().item())
print(
    "max |autograd - formula| dW_gate  :",
    (m.gate_proj.weight.grad - d_gate.T @ xg.detach()).abs().max().item(),
)
print("gradcheck:", torch.autograd.gradcheck(m, (xg.detach().clone().requires_grad_(),)))

## 8. Common implementation errors

- Swapping `gate_proj` and `up_proj` (SiLU applied to the wrong branch): shapes
  still match, outputs are wrong. Parity tests against HF catch it.
- Applying SiLU to the product, or to `up` as well.
- Adding biases: Qwen3's MLP has none (`bias=False`, independent of `attention_bias`).
- Transposing weights: `nn.Linear` stores `(out_features, in_features)`; the C++
  port must use `y = x @ W.T`.
- Using GELU (`hidden_act` is `"silu"`), or `torch.sigmoid(gate) * up` (that is GLU).
- Sizing `intermediate_size` as `4 * hidden_size`: Qwen3-0.6B uses 3072 = 3 × 1024.

## 9. Key lessons and review questions

1. Why does a gated FFN need three weight matrices instead of two?
2. What happens to the output when every `gate` value is very negative?
3. Why is SwiGLU not homogeneous of degree 1 (`f(2x) ≠ 2 f(x)`)?
4. How many parameters does the MLP add per layer in Qwen3-0.6B, and what share of the model is that?
5. Where does the `[B, T, I]` intermediate tensor dominate memory, and how could a C++ port avoid storing both `gate` and `up`?